# Week 3, day 2 — Worksheet 02 SOLUTIONS: MultiIndex   (L04)

Executed in the lab image (pandas 3.0.5) against the real
`data/orders_long.csv`. Every quoted number is what it actually printed.

Question 3 is the one to re-read. The level order decides what you can select
cheaply, and `groupby` chose it for you.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 02 — MultiIndex. Run this once.
import pandas as pd

orders = pd.read_csv("data/orders_long.csv")

# groupby on two columns -> a MultiIndex, without asking for one.
by_ry = orders.groupby(["Region", "Year"])["Sales"].sum().round(2)

print(by_ry.head(8))
print()
print("index type:", type(by_ry.index).__name__)

PART A — where it came from

### Question 1

names `['Region', 'Year']`, `nlevels 2`, `32` rows. -> level 0 lists 8 regions, level 1 lists 4 years.

The levels store the *distinct values* of each level — 8 and 4 — not the
32 combinations. That is how a MultiIndex stays compact: it holds two small
vocabularies plus codes saying which pair each row uses.

It also means `index.levels` can list a value that no row actually uses.
After filtering a MultiIndexed frame the levels are not automatically
pruned, so `levels[0]` may name regions that are no longer present.

In [ ]:
print("names:  ", by_ry.index.names)
print("nlevels:", by_ry.index.nlevels)
print("rows:   ", len(by_ry))
print()
print("level 0 values:", list(by_ry.index.levels[0]))
print("level 1 values:", list(by_ry.index.levels[1]))

### Question 2

Sorted by `Region` alphabetically, then `Year` ascending within each region — Atlantic first, Yukon last.

`groupby` sorted this without being asked. Your input file is in order of
order date; the output is in alphabetical order of region.

That is `sort=True`, the default. It costs time on large data and it is
occasionally wrong for you — month names sort as `Apr, Aug, Dec, Feb...`,
which is why `groupby("Month")` on text months produces a table nobody can
read. Pass `sort=False` to keep first-appearance order, or use a
`Categorical` with a real ordering.

In [ ]:
print(by_ry.to_string())

### Question 3

`by_ry.loc["Ontario"]` -> a **Series** indexed by `Year` alone. `by_ry.loc[("Ontario", 2012)]` -> `149578.59`.

Selecting on the outer level **consumes** it: the result is indexed by
`Year` only, because every remaining row is Ontario and the level no longer
distinguishes anything.

A tuple addresses both levels at once and reaches a single value. Note the
tuple needs its own parentheses — `by_ry.loc["Ontario", 2012]` means
something different to Pandas (row, column) and will confuse you on a
DataFrame.

In [ ]:
print("by_ry.loc['Ontario']:")
print(by_ry.loc["Ontario"])
print("-> type:", type(by_ry.loc["Ontario"]).__name__)
print("-> index name now:", by_ry.loc["Ontario"].index.name)
print()
print("one cell:", by_ry.loc[("Ontario", 2012)])

### Question 4

`by_ry.loc[2012]` -> **`KeyError: 2012`**. -> `by_ry.xs(2012, level="Year")` works and gives all 8 regions.

`.loc` on a MultiIndex reads left to right, so a bare key is matched
against the **outer** level only. `2012` is not a region, so it raises.

`.xs()` takes an explicit `level=`, so it can cut across the inner level.
The alternative is `by_ry.loc[(slice(None), 2012)]`, which is the same idea
written less readably.

The practical consequence is that **the order you pass to `groupby`
decides which questions are cheap later**. `groupby(["Region", "Year"])`
makes per-region selection natural and per-year selection awkward. Choose
it for how the result will be used, not for how the sentence sounds.

In [ ]:
try:
    print(by_ry.loc[2012])
except Exception as exc:
    print("by_ry.loc[2012] -> %s: %s" % (type(exc).__name__, str(exc)[:70]))

print()
print("by_ry.xs(2012, level='Year'):")
print(by_ry.xs(2012, level="Year").to_string())

### Question 5

After `swaplevel()` the rows read `2009 Atlantic, 2010 Atlantic, 2011 Atlantic...` — grouped by region still. After `.sort_index()` they read `2009 Atlantic, 2009 Northwest Territories, 2009 Nunavut...`.

`swaplevel()` changed the level *order* but not the row order, so the
result looks wrong: `Year` is the outer level and yet the years are
interleaved.

That is because swapping relabels the levels without re-sorting the rows.
Until you call `.sort_index()`, the index is not lexicographically sorted —
and selecting on an unsorted MultiIndex is both slower and, for slice
selections, an error.

**`swaplevel()` and `sort_index()` go together.** Treat the pair as one
operation.

In [ ]:
swapped = by_ry.swaplevel()
print("after swaplevel, unsorted:")
print(swapped.head(8).to_string())
print()
print("after .sort_index():")
print(swapped.sort_index().head(8).to_string())

PART B — getting back out

### Question 6

`reset_index()` -> a `(32, 3)` frame with columns `['Region', 'Year', 'Sales']` and a plain `RangeIndex`.

Both levels became ordinary columns and the index reverted to row numbers.
This is the long format from worksheet 01, and it is the shape to return to
whenever a MultiIndex is making a merge, an export or a plot awkward.

Note the Series became a DataFrame — `reset_index` has to put the values
somewhere, so the Series' name becomes a column.

In [ ]:
flat = by_ry.reset_index()
print(flat.head(6).to_string(index=False))
print()
print("shape:", flat.shape)
print("columns:", list(flat.columns))
print("index type now:", type(flat.index).__name__)

### Question 7

`set_index(["Region", "Year"])["Sales"]` -> identical to the original, `.equals()` and `index.equals()` both `True`.

A clean round trip. `reset_index` and `set_index` are exact inverses when
the levels are the same and in the same order, which makes it safe to flatten
for one operation and rebuild afterwards.

Swap the two names in `set_index` and the round trip still works but the
result is not equal — the level order is part of the identity.

In [ ]:
flat = by_ry.reset_index()
rebuilt = flat.set_index(["Region", "Year"])["Sales"]
print(rebuilt.head(4).to_string())
print()
print("same as the original:", rebuilt.equals(by_ry))
print("index equal:", rebuilt.index.equals(by_ry.index))

### Question 8

Collapsing by level and grouping the raw data directly -> **`identical: False`**. Difference: `Prarie -0.01`, `West -0.01`; the other six regions match exactly.

Six of eight reconcile and two are a penny out.

`by_ry` was rounded to 2dp when it was built, so collapsing it adds four
already-rounded numbers per region. The direct version adds the raw values
and rounds once at the end. For most regions the rounding errors cancelled;
for Prarie and West they did not.

This is worksheet 01 Q5 again in a different disguise, and it is the more
dangerous form — there is no export file to blame, just an intermediate
result that someone rounded for display and then reused as an input.

**Round for presentation, never for storage.** Keep full precision in
anything another step will consume, and round only in the final table a
human reads.

In [ ]:
collapsed = by_ry.groupby(level="Region").sum().round(2)
direct = orders.groupby("Region")["Sales"].sum().round(2)
print(collapsed.to_string())
print()
print("identical:", collapsed.equals(direct))
print()
diff = (collapsed - direct).round(4)
print("difference per region:")
print(diff[diff != 0].to_string() if (diff != 0).any() else "  none")

# by_ry was ROUNDED to 2dp when it was built, so this sums 4 rounded
# numbers per region. The direct version rounds the raw sum once.

### Question 9

`90` rows against `96` possible combinations -> **6 missing**.

Adding a third level multiplied the possible cells to 8 x 4 x 3 = 96, and
the data only fills 90 of them. Worksheet 01 showed the two-level version
was complete at 32 of 32.

That is the general shape: **the cells grow multiplicatively and the data
does not.** Every extra dimension you group by makes the result sparser,
and a sparse grid is where surprising `NaN`s come from after a reshape.

In [ ]:
three = orders.groupby(["Region", "Year", "Category"])["Sales"].sum().round(2)
possible = (orders["Region"].nunique() * orders["Year"].nunique()
            * orders["Category"].nunique())
print("rows:", len(three))
print("possible combinations:", possible)
print("missing combinations:  ", possible - len(three))
print()
print(three.head(6).to_string())

### Question 10

`by_ry.loc[("Nunavut", 2013)]` -> **raises** `KeyError: ('Nunavut', 2013)`. -> years present are `[2009, 2010, 2011, 2012]`.

Both halves of the tuple are checked, and the message names the whole pair
rather than telling you which half was wrong. On a real index with hundreds
of values that is worth remembering: print
`index.get_level_values(name).unique()` for each level to find out which one
is missing.

Nunavut exists, 2013 does not. The error would look identical if it were
the other way round.

In [ ]:
print("years present:", sorted(by_ry.index.get_level_values("Year").unique()))
print(by_ry.loc[("Nunavut", 2013)])